In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters pypdf chromadb sentence-transformers networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203

In [2]:
import os
import torch
import requests
import networkx as nx
import pickle
import shutil
from google.colab import drive
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Mount Drive
drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System is running on {device.upper()}")



Mounted at /content/drive
System is running on CUDA


In [3]:
# Define the Multi-Source Library
pdf_sources = {
    "NIST_800-207": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-207.pdf",
    "NIST_800-53": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf",
    "CISA_ZT_Maturity_v2": "https://www.cisa.gov/sites/default/files/2023-04/zero_trust_maturity_model_v2_508.pdf",
    "NSA_ZT_Data_Pillar": "https://media.defense.gov/2024/Apr/09/2003434442/-1/-1/0/CSI_DATA_PILLAR_ZT.PDF"
}

all_raw_documents = []
os.makedirs("zt_docs", exist_ok=True)

# Spoof the user agent to bypass basic bot blockers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print("Downloading and reading PDF frameworks...")
for name, url in pdf_sources.items():
    file_path = f"zt_docs/{name}.pdf"

    try:
        response = requests.get(url, headers=headers, timeout=15)

        # Verify it's actually a PDF
        content_type = response.headers.get('Content-Type', '')
        if 'pdf' not in content_type.lower() and response.content[:4] != b'%PDF':
            print(f"  [!] Blocked: {name} returned HTML instead of a PDF. Skipping...")
            continue

        with open(file_path, 'wb') as f:
            f.write(response.content)

        loader = PyPDFLoader(file_path)
        docs = loader.load()

        # Inject metadata for citations
        for doc in docs:
            doc.metadata['source_document'] = name

        all_raw_documents.extend(docs)
        print(f"  -> Successfully loaded {len(docs)} pages from {name}")

    except Exception as e:
        print(f"  [!] Failed to process {name}: {e}")

print(f"\nTotal Pages Loaded: {len(all_raw_documents)}")

  -> Successfully loaded 59 pages from NIST_800-207
  -> Successfully loaded 492 pages from NIST_800-53
  -> Successfully loaded 32 pages from CISA_ZT_Maturity_v2
  [!] Blocked: NSA_ZT_Data_Pillar returned HTML instead of a PDF. Skipping...

Total Pages Loaded: 583


In [4]:
# The Aggressive Chunking Strategy to hit >2,000 chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(all_raw_documents)
print(f"Developer Log: Sliced documents into {len(chunks)} individual knowledge chunks.")

if len(chunks) > 2000:
    print(f"STATUS: BENCHMARK MET! ({len(chunks)} chunks successfully generated)")
else:
    print(f"STATUS: WARNING - Did not hit 2000 chunk benchmark. Only generated {len(chunks)} chunks.")

Developer Log: Sliced documents into 4248 individual knowledge chunks.
STATUS: BENCHMARK MET! (4248 chunks successfully generated)


In [5]:
print("Building Knowledge Graph...")
G = nx.Graph()

zta_entities = [
    "Policy Decision Point", "PDP", "Policy Enforcement Point", "PEP",
    "Policy Administrator", "Trust Algorithm", "Data Plane", "Control Plane",
    "Micro-segmentation", "Continuous Diagnostics and Mitigation", "CDM"
]

for i, chunk in enumerate(chunks):
    chunk_id = f"chunk_{i}"
    doc_source = chunk.metadata.get('source_document', 'Unknown')

    G.add_node(chunk_id, type="chunk", text=chunk.page_content)
    G.add_node(doc_source, type="document")
    G.add_edge(chunk_id, doc_source, relation="EXTRACTED_FROM")

    text_lower = chunk.page_content.lower()
    for entity in zta_entities:
        if entity.lower() in text_lower:
            G.add_node(entity, type="concept")
            G.add_edge(chunk_id, entity, relation="MENTIONS")

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Building Knowledge Graph...
Graph created with 4262 nodes and 4383 edges.


In [6]:
print("Generating BGE Embeddings (This will take a few minutes)...")
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {'device': device}
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Initialize ChromaDB
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_bge"
)

print("Developer Log: Massive Vector Database successfully indexed.")

Generating BGE Embeddings (This will take a few minutes)...


/tmp/ipykernel_3536/2398435280.py:6: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Developer Log: Massive Vector Database successfully indexed.


In [7]:
drive_path = '/content/drive/My Drive/ZTA_Project'
os.makedirs(drive_path, exist_ok=True)

# Save Vector DB
shutil.copytree("./chroma_db_bge", f"{drive_path}/chroma_db_bge", dirs_exist_ok=True)

# Save Knowledge Graph
with open(f"{drive_path}/zta_knowledge_graph.gpickle", 'wb') as f:
    pickle.dump(G, f)

print(f"SUCCESS! Phase 1 complete. Database and Graph saved to {drive_path}")

SUCCESS! Phase 1 complete. Database and Graph saved to /content/drive/My Drive/ZTA_Project
